In [16]:
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

--2026-09-15 09:55:39--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 2606:50c0:8003::154, 2606:50c0:8001::154, 2606:50c0:8002::154, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|2606:50c0:8003::154|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt.1’

input.txt.1         100%[===================>]   1.06M  1.35MB/s    in 0.8s    

2026-09-15 09:55:40 (1.35 MB/s) - ‘input.txt.1’ saved [1115394/1115394]



In [17]:
with open("input.txt") as f:
    text = f.read()

In [18]:
print(len(text))

1115394


In [19]:
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(''.join(chars))
print(vocab_size)


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
65


In [20]:
stoi = {ch:i for i, ch in enumerate(chars)}
itos = {i:ch for i, ch in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])

In [21]:
s = "hii there!"
print(encode(s))
print(decode(encode(s)))

[46, 47, 47, 1, 58, 46, 43, 56, 43, 2]
hii there!


In [22]:
import torch
data = torch.tensor(encode(text), dtype=torch.long)
print(data.shape, data.dtype)

torch.Size([1115394]) torch.int64


In [23]:
n = int(0.9*len(data))
train_data = data[:n]
val_data = data[n:]

In [24]:
print(data[:8])
print(data[8:16])

tensor([18, 47, 56, 57, 58,  1, 15, 47])
tensor([58, 47, 64, 43, 52, 10,  0, 14])


In [25]:
block_size = 8
x = train_data[:block_size]
y = train_data[1:block_size + 1]

print(x, y, sep='\n')

for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print(f"when input is {context}, the target is: {target}")

tensor([18, 47, 56, 57, 58,  1, 15, 47])
tensor([47, 56, 57, 58,  1, 15, 47, 58])
when input is tensor([18]), the target is: 47
when input is tensor([18, 47]), the target is: 56
when input is tensor([18, 47, 56]), the target is: 57
when input is tensor([18, 47, 56, 57]), the target is: 58
when input is tensor([18, 47, 56, 57, 58]), the target is: 1
when input is tensor([18, 47, 56, 57, 58,  1]), the target is: 15
when input is tensor([18, 47, 56, 57, 58,  1, 15]), the target is: 47
when input is tensor([18, 47, 56, 57, 58,  1, 15, 47]), the target is: 58


In [26]:
torch.manual_seed(1234)

batch_size = 4
block_size = 8

def get_batch(split):
    data = train_data if split == "train" else val_data
    ix = torch.randint(len(data) - block_size, (batch_size, ))
    x = torch.stack([data[i:i + block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size + 1] for i in ix])
    return x, y

xb, yb = get_batch("train")
print(xb.shape, xb, yb, sep='\n')
print('------')

for b in range(batch_size):
    for t in range(block_size):
        context = xb[b][:t+1]
        target =  yb[b][t]
        print(f"context: {context}, target: {target}")

torch.Size([4, 8])
tensor([[51, 59, 57, 58,  1, 39, 61, 39],
        [63,  6,  1, 61, 46, 53,  5, 57],
        [39,  1, 41, 46, 47, 50, 42, 10],
        [53, 59, 10,  0, 37, 53, 59,  1]])
tensor([[59, 57, 58,  1, 39, 61, 39, 63],
        [ 6,  1, 61, 46, 53,  5, 57,  1],
        [ 1, 41, 46, 47, 50, 42, 10,  0],
        [59, 10,  0, 37, 53, 59,  1, 57]])
------
context: tensor([51]), target: 59
context: tensor([51, 59]), target: 57
context: tensor([51, 59, 57]), target: 58
context: tensor([51, 59, 57, 58]), target: 1
context: tensor([51, 59, 57, 58,  1]), target: 39
context: tensor([51, 59, 57, 58,  1, 39]), target: 61
context: tensor([51, 59, 57, 58,  1, 39, 61]), target: 39
context: tensor([51, 59, 57, 58,  1, 39, 61, 39]), target: 63
context: tensor([63]), target: 6
context: tensor([63,  6]), target: 1
context: tensor([63,  6,  1]), target: 61
context: tensor([63,  6,  1, 61]), target: 46
context: tensor([63,  6,  1, 61, 46]), target: 53
context: tensor([63,  6,  1, 61, 46, 53]), ta

In [27]:
print(xb) #batch input

tensor([[51, 59, 57, 58,  1, 39, 61, 39],
        [63,  6,  1, 61, 46, 53,  5, 57],
        [39,  1, 41, 46, 47, 50, 42, 10],
        [53, 59, 10,  0, 37, 53, 59,  1]])


In [31]:
import torch
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1234)

class BigramLanguageModel(nn.Module):
    
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)
        
    def forward(self, idx, targets=None):
        logits = self.token_embedding_table(idx) # B T C
        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
    
        return logits, loss
    
    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            logits, loss = self(idx)
            logits = logits[:, -1, :] # becomes (B, C)
            probs = F.softmax(logits, dim=-1) # (B, C)
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx
    
m = BigramLanguageModel(vocab_size)
logits, loss = m(xb, yb)
print(logits.shape)
print(loss)

print(decode(m.generate(torch.zeros((1,1), dtype=torch.long), max_new_tokens=10)[0].tolist()))

torch.Size([32, 65])
tensor(4.8650, grad_fn=<NllLossBackward0>)

UkBP-NBU& 


In [36]:
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)

In [44]:
batch_size = 32

for _ in range(1000):
    xb, yb = get_batch('train')
    logits, loss = m(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
    
print(loss)
    

tensor(2.3832, grad_fn=<NllLossBackward0>)


In [57]:
s = "To be or not to be, that is the "
idx = torch.tensor([encode(s)], dtype=torch.long)
print(decode(m.generate(idx, 100)[0].tolist()))

To be or not to be, that is the andethad IAMiseerh y, t! we, wise;
ake ch f the ar3ghe?
Anold cou heath MERLO atoug:
Leinsthe.NROXUT
